In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
from joblib import Parallel, delayed

from pd_estim_A.data.data_import import (
    load_data, load_ecb_1y_yield,
    fill_liabilities, drop_high_leverage_firms,
    prepare_merton_inputs
)
from pd_estim_A.models.merton.merton_main import (
    process_one_firm_merton,
)

from pd_estim_A.data.cds_df import get_cds_panel

In [3]:
# Data processing and df preparation
print(Path.cwd())
data_path = Path.cwd() / ".." / "data" / "raw"
output_path = Path.cwd() / ".." / "data" / "derived"
ret_daily, bs, coverage = load_data(
    data_path / "Jan2025_Accenture_Dataset_ErasmusCase.xlsx",
    start_date="2012-01-01",
    end_date="2025-12-19",
    enforce_coverage=True,
    coverage_tol=0.995,
    liabilities_scale="auto",
    verbose=True,
)

df_rf = load_ecb_1y_yield(
    startPeriod="2010-01-01",
    endPeriod="2025-12-31",
    out_file= output_path / "ecb_yc_1y_aaa.xml",
    verify_ssl=True,  # recommended if it works
)

df_cal = ret_daily[["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)

debt_daily = fill_liabilities(bs, df_cal)

ret_filt, bs_filt, lev_by_firm, dropped = drop_high_leverage_firms(
    ret_daily,
    bs,
    df_calendar=df_cal,
    debt_daily=debt_daily,
    lev_threshold=8.0,
    lev_agg="median",
    verbose=True,
)

# keep debt panel consistent with filtered firms
keep = set(ret_filt["gvkey"].astype(str).unique())
debt_daily_filt = debt_daily[debt_daily["gvkey"].astype(str).isin(keep)].copy()

# Merton
merton_df = prepare_merton_inputs(ret_filt, bs_filt, df_rf, debt_daily=debt_daily_filt)

c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test
[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
Data has been written to c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\ecb_yc_1y_aaa.xml
[drop_high_leverage_firms] agg=median, threshold=8.0
[drop_high_leverage_firms] firms before: 46 | after: 36
[drop_high_leverage_firms] dropped firms: 10


In [4]:
# call cds panel and merge
cds = get_cds_panel(
    project_root= Path.cwd() / "..",
    save_csv=False,
    verbose=True,
)
# ensure types
merton = merton_df.copy()
merton["gvkey"] = merton["gvkey"].astype(str)
merton["date"]  = pd.to_datetime(merton["date"])

cds["gvkey"] = cds["gvkey"].astype(str)
cds["date"]  = pd.to_datetime(cds["date"])

# keep only firms that exist in BOTH (drop firms with no CDS)
common_gv = sorted(set(merton["gvkey"].unique()) & set(cds["gvkey"].unique()))
merton = merton[merton["gvkey"].isin(common_gv)].copy()
cds = cds[cds["gvkey"].isin(common_gv)].copy()

# also drop CDS rows whose dates are outside merged's date range
dmin, dmax = merton["date"].min(), merton["date"].max()
cds = cds[(cds["date"] >= dmin) & (cds["date"] <= dmax)].copy()

# merge-asof onto merged's dates (direction='backward')
merton = merton.sort_values(["date", "gvkey"]).reset_index(drop=True)
cds    = cds.sort_values(["date", "gvkey"]).reset_index(drop=True)

merged_cds = pd.merge_asof(
    merton,
    cds,
    on="date",
    by="gvkey",
    direction="backward",
    allow_exact_matches=True,
)

# drop rows where CDS still missing
merton_df = merged_cds.dropna(subset=["cds"]).reset_index(drop=True)

print("firms after intersection:", merton_df["gvkey"].nunique())
print("rows after merge:", len(merton_df))
print("date range:", merton_df["date"].min(), "→", merton_df["date"].max())

[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
[get_cds_panel] sheets read: 22 | rows parsed: 67015 | unmapped sheets: 0
firms after intersection: 21
rows after merge: 65569
date range: 2014-01-01 00:00:00 → 2025-12-19 00:00:00


In [5]:
# Rolling configuration
TRAIN_YEARS = 2
STEP_FREQ = "QE"
WEEK_ENDING = "W-FRI"
T_HORIZON = 1.0               # KMV-style horizon for DD/PD
DATA_END = pd.Timestamp("2024-12-31")  # stop at end of dataset
MIN_DAILY_ROWS = 10

# OOS definition: "next quarter"
# We'll run windows where the OOS quarter end is <= DATA_END.
# So the last train_end will be the quarter end immediately before DATA_END.
LAST_TRAIN_END = (DATA_END - pd.offsets.QuarterEnd(1))

# Debug / runtime control: start small, then scale up.
MAX_FIRMS = None      # e.g. 2 for quick test; None for all firms
MAX_WINDOWS = None # e.g. 2 for quick test; None for all windows

# One-time preprocessing for speed
panel = merton_df.copy()
panel["gvkey"] = panel["gvkey"].astype(str)
panel["date"] = pd.to_datetime(panel["date"])

# Keep only columns we need repeatedly
needed_cols = ["gvkey","date","company","E","B","r","sigma_E"]
panel = panel[[c for c in needed_cols if c in panel.columns]].copy()

# Coerce numeric once
for c in ["E","B","r","sigma_E"]:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

# Add constants once
panel["T"] = float(T_HORIZON)

# Basic cleaning
panel = (
    panel.dropna(subset=["date","E","B","r","T"])
         .query("E > 0 and B > 0")
         .sort_values(["gvkey","date"])
)

# Build per-firm daily panels once
firm_daily = {}
for gvkey, g in panel.groupby("gvkey", sort=False):
    g = g.sort_values("date")
    # dedupe dates if needed
    g = g.groupby("date", as_index=False).last()
    firm_daily[gvkey] = g.set_index("date")

gvkeys_all = sorted(firm_daily.keys())
if MAX_FIRMS is not None:
    gvkeys_all = gvkeys_all[:int(MAX_FIRMS)]

print("Firms loaded:", len(firm_daily), "| Firms in run:", len(gvkeys_all))
print("Panel date range:", panel["date"].min().date(), "to", panel["date"].max().date())
print("LAST_TRAIN_END:", LAST_TRAIN_END.date(), "| DATA_END:", DATA_END.date())

Firms loaded: 21 | Firms in run: 21
Panel date range: 2014-01-01 to 2025-12-19
LAST_TRAIN_END: 2024-09-30 | DATA_END: 2024-12-31


In [6]:
# Build the rolling quarter schedule
global_min_date = panel["date"].min()

# earliest possible train_end is >= (min_date + TRAIN_YEARS - 1 day), aligned to quarter-ends
earliest_end = (global_min_date + pd.DateOffset(years=TRAIN_YEARS) - pd.Timedelta(days=1))

train_ends = pd.date_range(start=earliest_end, end=LAST_TRAIN_END, freq=STEP_FREQ)
train_ends = pd.to_datetime(train_ends)

if MAX_WINDOWS is not None:
    train_ends = train_ends[:int(MAX_WINDOWS)]

windows = []
for train_end in train_ends:
    train_start = train_end - pd.DateOffset(years=TRAIN_YEARS) + pd.Timedelta(days=1)
    oos_start = train_end + pd.Timedelta(days=1)
    oos_end = train_end + pd.offsets.QuarterEnd(1)  # next quarter end

    windows.append({
        "train_start": pd.Timestamp(train_start),
        "train_end": pd.Timestamp(train_end),
        "oos_start": pd.Timestamp(oos_start),
        "oos_end": pd.Timestamp(oos_end),
    })

windows_df = pd.DataFrame(windows)
windows_df

,train_start,train_end,oos_start,oos_end
0,2014-01-01,2015-12-31,2016-01-01,2016-03-31
1,2014-04-01,2016-03-31,2016-04-01,2016-06-30
2,2014-07-01,2016-06-30,2016-07-01,2016-09-30
3,2014-10-01,2016-09-30,2016-10-01,2016-12-31
4,2015-01-01,2016-12-31,2017-01-01,2017-03-31
5,2015-04-01,2017-03-31,2017-04-01,2017-06-30
6,2015-07-01,2017-06-30,2017-07-01,2017-09-30
7,2015-10-01,2017-09-30,2017-10-01,2017-12-31
8,2016-01-01,2017-12-31,2018-01-01,2018-03-31
9,2016-04-01,2018-03-31,2018-04-01,2018-06-30


In [ ]:
# Rolling estimation + PDs + OOS inversion (parallelized across firms)

# Optional: keep a small test mode if desired
gvkeys_run = gvkeys_all[:int(MAX_FIRMS)] if MAX_FIRMS is not None else gvkeys_all
windows_run = windows[:int(MAX_WINDOWS)] if MAX_WINDOWS is not None else windows

print(f"Running {len(gvkeys_run)} firms across {len(windows_run)} windows")
print("Parallelization: across firms | sequential within each firm")

# One worker per firm
parallel_results = Parallel(
    n_jobs=-1,          # use all available cores
    backend="loky",     # process-based parallelism
    verbose=10,
)(
    delayed(process_one_firm_merton)(
        firm_daily[gvkey],
        windows=windows_run,
        gvkey=gvkey,
        date_col="date",
        week_ending=WEEK_ENDING,
        ann_factor=52.0,
        T_horizon=T_HORIZON,
        min_daily_rows=MIN_DAILY_ROWS,
        min_weekly_obs=None,
        min_weekly_returns=2,
        E_col="E",
        B_col="B",
        r_col="r",
        T_col="T",
        B_scale=1.0,
        sigmaE_col="sigma_E" if "sigma_E" in panel.columns else None,
    )
    for gvkey in gvkeys_run
)

# Unpack firm-level outputs
summary_parts = []
weekly_is_parts = []
weekly_oos_parts = []

for summary_df_i, weekly_is_df_i, weekly_oos_df_i in parallel_results:
    if summary_df_i is not None and not summary_df_i.empty:
        summary_parts.append(summary_df_i)
    if weekly_is_df_i is not None and not weekly_is_df_i.empty:
        weekly_is_parts.append(weekly_is_df_i)
    if weekly_oos_df_i is not None and not weekly_oos_df_i.empty:
        weekly_oos_parts.append(weekly_oos_df_i)

# Final rolling outputs, same structure as before
roll_summary_df = (
    pd.concat(summary_parts, ignore_index=True)
      .sort_values(["train_end", "gvkey"])
      .reset_index(drop=True)
) if len(summary_parts) else pd.DataFrame()

roll_weekly_is_df = (
    pd.concat(weekly_is_parts, ignore_index=True)
      .sort_values(["train_end", "gvkey", "date"])
      .reset_index(drop=True)
) if len(weekly_is_parts) else pd.DataFrame()

roll_weekly_oos_df = (
    pd.concat(weekly_oos_parts, ignore_index=True)
      .sort_values(["train_end", "gvkey", "date"])
      .reset_index(drop=True)
) if len(weekly_oos_parts) else pd.DataFrame()

roll_summary_df, roll_weekly_is_df, roll_weekly_oos_df

Running 21 firms across 36 windows
Parallelization: across firms | sequential within each firm


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of  21 | elapsed:    9.2s remaining:   39.3s
[Parallel(n_jobs=-1)]: Done   7 out of  21 | elapsed:   13.4s remaining:   27.0s
[Parallel(n_jobs=-1)]: Done  10 out of  21 | elapsed:   13.6s remaining:   15.0s
[Parallel(n_jobs=-1)]: Done  13 out of  21 | elapsed:   14.6s remaining:    9.0s
[Parallel(n_jobs=-1)]: Done  16 out of  21 | elapsed:   19.9s remaining:    6.1s
[Parallel(n_jobs=-1)]: Done  19 out of  21 | elapsed:   21.4s remaining:    2.2s
[Parallel(n_jobs=-1)]: Done  21 out of  21 | elapsed:   21.8s finished


(      gvkey train_start  train_end  oos_start    oos_end    ok  \
 0    100022  2014-01-01 2015-12-31 2016-01-01 2016-03-31  True   
 1    100080  2014-01-01 2015-12-31 2016-01-01 2016-03-31  True   
 2    100957  2014-01-01 2015-12-31 2016-01-01 2016-03-31  True   
 3    101202  2014-01-01 2015-12-31 2016-01-01 2016-03-31  True   
 4    101204  2014-01-01 2015-12-31 2016-01-01 2016-03-31  True   
 ..      ...         ...        ...        ...        ...   ...   
 751  222379  2022-10-01 2024-09-30 2024-10-01 2024-12-31  True   
 752   23667  2022-10-01 2024-09-30 2024-10-01 2024-12-31  True   
 753   23671  2022-10-01 2024-09-30 2024-10-01 2024-12-31  True   
 754  241456  2022-10-01 2024-09-30 2024-10-01 2024-12-31  True   
 755   61616  2022-10-01 2024-09-30 2024-10-01 2024-12-31  True   
 
                  msg  sigma_hat  mu_hat_train         B_end pd_date_is  \
 0    converged(it=1)   0.104497      0.080798  1.173660e+11 2015-12-31   
 1    converged(it=1)   0.210482      0.1366

In [8]:
# Sanity checks and previews
print("roll_summary_df shape:", roll_summary_df.shape)
print("roll_weekly_is_df shape:", roll_weekly_is_df.shape)
print("roll_weekly_oos_df shape:", roll_weekly_oos_df.shape)

# Check that we have rolling windows
print("Unique train_end windows:", roll_summary_df["train_end"].nunique() if "train_end" in roll_summary_df.columns else None)

# Validity checks
if not roll_summary_df.empty:
    print("Share ok:", roll_summary_df["ok"].mean() if "ok" in roll_summary_df.columns else None)
    print("Sigma summary (ok only):")
    print(roll_summary_df.loc[roll_summary_df["ok"] == True, "sigma_hat"].describe())

# Peek one firm across time
if not roll_summary_df.empty:
    g0 = roll_summary_df.loc[roll_summary_df["ok"] == True, "gvkey"].astype(str).unique()
    if len(g0):
        example = g0[0]
        display(
            roll_summary_df[(roll_summary_df["gvkey"] == example) & (roll_summary_df["ok"] == True)]
            [["gvkey","train_end","sigma_hat","mu_hat_train","PD_Q_1y_is","PD_P_1y_is"]]
            .head(15)
        )

# Peek OOS PDs for same firm
if not roll_weekly_oos_df.empty and len(g0):
    display(
        roll_weekly_oos_df[roll_weekly_oos_df["gvkey"] == example]
        [["gvkey","train_end","date","sigma_hat","PD_Q_1y_oos","PD_P_1y_oos"]]
        .head(30)
    )

roll_summary_df shape: (756, 16)
roll_weekly_is_df shape: (79296, 9)
roll_weekly_oos_df shape: (10206, 13)
Unique train_end windows: 36
Share ok: 1.0
Sigma summary (ok only):
count    756.000000
mean       0.154294
std        0.061487
min        0.050427
25%        0.109240
50%        0.150426
75%        0.182165
max        0.360777
Name: sigma_hat, dtype: float64


,gvkey,train_end,sigma_hat,mu_hat_train,PD_Q_1y_is,PD_P_1y_is
0,100022,2015-12-31,0.104497,0.080798,0.000067,1.816869e-06
21,100022,2016-03-31,0.123361,0.065403,0.006035,1.036314e-03
42,100022,2016-06-30,0.124684,0.037266,0.019621,7.919892e-03
63,100022,2016-09-30,0.123076,0.073102,0.009452,1.349783e-03
84,100022,2016-12-31,0.122820,0.090438,0.003144,2.033588e-04
105,100022,2017-03-31,0.118091,0.024873,0.005300,2.333339e-03
126,100022,2017-06-30,0.113388,0.043866,0.005359,1.366889e-03
147,100022,2017-09-30,0.108377,0.087203,0.002519,1.173183e-04
168,100022,2017-12-31,0.103671,0.053587,0.001503,1.888117e-04
189,100022,2018-03-31,0.074950,0.046029,0.000011,3.520980e-07


,gvkey,train_end,date,sigma_hat,PD_Q_1y_oos,PD_P_1y_oos
0,100022,2015-12-31,2016-01-01,0.104497,0.000067,0.000002
1,100022,2015-12-31,2016-01-08,0.104497,0.000412,0.000045
2,100022,2015-12-31,2016-01-15,0.104497,0.000796,0.000142
3,100022,2015-12-31,2016-01-22,0.104497,0.000650,0.000102
4,100022,2015-12-31,2016-01-29,0.104497,0.000939,0.000189
5,100022,2015-12-31,2016-02-05,0.104497,0.001548,0.000433
6,100022,2015-12-31,2016-02-12,0.104497,0.001987,0.000645
7,100022,2015-12-31,2016-02-19,0.104497,0.001316,0.000333
8,100022,2015-12-31,2016-02-26,0.104497,0.001308,0.000331
9,100022,2015-12-31,2016-03-04,0.104497,0.000459,0.000060


In [9]:
# Show in-sample PDs and the exact V and B used (and r) next to them
# last weekly implied asset row in-sample per (train_end, gvkey)
is_last = (
    roll_weekly_is_df.sort_values(["train_end", "gvkey", "date"])
    .groupby(["train_end", "gvkey"], as_index=False)
    .tail(1)
    .rename(columns={"V_hat": "V_used_is", "r": "r_used_is"})
    [["train_end", "gvkey", "V_used_is", "r_used_is"]]
)

insample_view = (
    roll_summary_df
    .merge(is_last, on=["train_end", "gvkey"], how="left")
)

# add percent versions for readability
insample_view["PD_Q_1y_is_pct"] = 100.0 * insample_view["PD_Q_1y_is"]
insample_view["PD_P_1y_is_pct"] = 100.0 * insample_view["PD_P_1y_is"]

cols = [
    "gvkey", "train_end",
    "V_used_is", "B_end", "r_used_is",
    "sigma_hat", "mu_hat_train",
    "PD_Q_1y_is", "PD_Q_1y_is_pct",
    "PD_P_1y_is", "PD_P_1y_is_pct",
]

display(insample_view[cols].sort_values(["gvkey", "train_end"]).head())

,gvkey,train_end,V_used_is,B_end,r_used_is,sigma_hat,mu_hat_train,PD_Q_1y_is,PD_Q_1y_is_pct,PD_P_1y_is,PD_P_1y_is_pct
0,100022,2015-12-31,1.766053e+11,1.173660e+11,-0.003968,0.104497,0.080798,0.000067,0.006669,0.000002,0.000182
21,100022,2016-03-31,1.785910e+11,1.294100e+11,-0.004857,0.123361,0.065403,0.006035,0.603513,0.001036,0.103631
42,100022,2016-06-30,1.697459e+11,1.294100e+11,-0.006491,0.124684,0.037266,0.019621,1.962096,0.007920,0.791989
63,100022,2016-09-30,1.753335e+11,1.294100e+11,-0.007218,0.123076,0.073102,0.009452,0.945187,0.001350,0.134978
84,100022,2016-12-31,1.838910e+11,1.294100e+11,-0.008224,0.122820,0.090438,0.003144,0.314398,0.000203,0.020336


In [10]:
roll_summary_df = roll_summary_df.copy()
roll_weekly_is_df = roll_weekly_is_df.copy()
roll_weekly_oos_df = roll_weekly_oos_df.copy()

for df_ in [roll_summary_df, roll_weekly_is_df, roll_weekly_oos_df]:
    if "gvkey" in df_.columns:
        df_["gvkey"] = df_["gvkey"].astype(str)
    if "date" in df_.columns:
        df_["date"] = pd.to_datetime(df_["date"])
    if "train_end" in df_.columns:
        df_["train_end"] = pd.to_datetime(df_["train_end"])


# TRAINING-END rows
is_sorted = roll_weekly_is_df.sort_values(["gvkey", "train_end", "date"])

is_first = (
    is_sorted.groupby(["gvkey", "train_end"], as_index=False)
    .head(1)[["gvkey", "train_end", "V_hat"]]
    .rename(columns={"V_hat": "V_0"})
)

is_last = (
    is_sorted.groupby(["gvkey", "train_end"], as_index=False)
    .tail(1)[["gvkey", "train_end", "date", "V_hat"]]
    .rename(columns={"V_hat": "V_used"})
)

te = (
    roll_summary_df.loc[roll_summary_df["ok"] == True].copy()
    .merge(is_first, on=["gvkey", "train_end"], how="left")
    .merge(is_last,  on=["gvkey", "train_end"], how="left")
)

train_end_rows = pd.DataFrame({
    "gvkey": te["gvkey"].astype(str),
    "date": pd.to_datetime(te["date"]),
    "sigma_hat": te["sigma_hat"].astype(float),
    "mu_hat": te["mu_hat_train"].astype(float),
    "V_0": te["V_0"].astype(float),
    "V_used": te["V_used"].astype(float),
    "B_used": te["B_end"].astype(float),
    "PD_Q": te["PD_Q_1y_is"].astype(float),
    "PD_P": te["PD_P_1y_is"].astype(float),
    "train_end_date": pd.to_datetime(te["train_end"]),   # <-- added
    "training_end": 1,
})


# OOS weekly rows
required_oos_cols = ["gvkey","date","sigma_hat","mu_hat_oos_expanding","V_hat_oos","B_used","PD_Q_1y_oos","PD_P_1y_oos","train_end"]
missing = [c for c in required_oos_cols if c not in roll_weekly_oos_df.columns]
if missing:
    raise ValueError(f"roll_weekly_oos_df is missing required columns: {missing}")

oos = roll_weekly_oos_df.copy()

oos_rows = pd.DataFrame({
    "gvkey": oos["gvkey"].astype(str),
    "date": pd.to_datetime(oos["date"]),
    "sigma_hat": oos["sigma_hat"].astype(float),
    "mu_hat": oos["mu_hat_oos_expanding"].astype(float),
    "V_0": 0.0,
    "V_used": oos["V_hat_oos"].astype(float),
    "B_used": oos["B_used"].astype(float),
    "PD_Q": oos["PD_Q_1y_oos"].astype(float),
    "PD_P": oos["PD_P_1y_oos"].astype(float),
    "train_end_date": pd.to_datetime(oos["train_end"]),  # <-- added
    "training_end": 0,
})

# Combine
final_df = pd.concat([train_end_rows, oos_rows], ignore_index=True)

final_df = (
    final_df.sort_values(["gvkey", "date", "training_end"], ascending=[True, True, False])
            .drop_duplicates(subset=["gvkey", "date"], keep="first")
            .sort_values(["gvkey", "date"])
            .reset_index(drop=True)
)

final_df["training_end"] = final_df["training_end"].astype(int)

final_df = final_df[
    ["gvkey","date","sigma_hat","mu_hat","V_0","V_used","B_used","PD_Q","PD_P","train_end_date","training_end"]
]

final_df

,gvkey,date,sigma_hat,mu_hat,V_0,V_used,B_used,PD_Q,PD_P,train_end_date,training_end
0,100022,2015-12-31,0.104497,0.080798,1.519029e+11,1.766053e+11,1.173660e+11,6.669229e-05,0.000002,2015-12-31,1
1,100022,2016-01-01,0.104497,0.080080,0.000000e+00,1.766053e+11,1.173660e+11,6.669224e-05,0.000002,2015-12-31,0
2,100022,2016-01-08,0.104497,0.055184,0.000000e+00,1.681074e+11,1.173660e+11,4.121455e-04,0.000045,2015-12-31,0
3,100022,2016-01-15,0.104497,0.045133,0.000000e+00,1.648239e+11,1.173660e+11,7.961513e-04,0.000142,2015-12-31,0
4,100022,2016-01-22,0.104497,0.047777,0.000000e+00,1.658577e+11,1.173660e+11,6.502815e-04,0.000102,2015-12-31,0
...,...,...,...,...,...,...,...,...,...,...,...
10222,61616,2024-12-06,0.085065,-0.006505,0.000000e+00,1.310894e+11,8.896200e+10,8.966219e-07,0.000005,2024-09-30,0
10223,61616,2024-12-13,0.085065,-0.007816,0.000000e+00,1.306842e+11,8.896200e+10,1.075185e-06,0.000006,2024-09-30,0
10224,61616,2024-12-20,0.085065,-0.014081,0.000000e+00,1.288423e+11,8.896200e+10,2.474818e-06,0.000017,2024-09-30,0
10225,61616,2024-12-27,0.085065,-0.011380,0.000000e+00,1.295835e+11,8.896200e+10,1.773760e-06,0.000011,2024-09-30,0


In [11]:
# save final_df as CSV in the current working directory
output_path = Path.cwd() / ".." / "data" / "derived"
final_df.to_csv(output_path / "merton_weekly.csv", index=False)